# Predicción del Precio de Bolsa — Mercado Eléctrico Colombiano
## Entrega 1: Problema, Datos, EDA y Baseline

**Curso:** Aprendizaje de Máquina Aplicado  
**Profesor:** Marco Terán — Universidad EAFIT  
**Fecha:** Abril 2026

---

## Tabla de contenidos

1. [Definición del problema](#problema)
2. [Setup y dependencias](#setup)
3. [Carga de datos](#carga)
4. [Auditoría de calidad](#calidad)
5. [EDA — Análisis del target](#target)
6. [EDA — Variables: relaciones, señal predictiva y selección](#eda)
   - 6a. Relaciones entre variables
   - 6b. Relaciones con el target
   - 6c. Evaluación de reducción de features
7. [EDA — Fenómeno El Niño 2023-2024](#nino)
8. [Análisis de redundancia de columnas originales](#redundancia)
9. [Feature engineering — 5 features justificadas](#engineering)
10. [Partición temporal](#split)
11. [Baseline reproducible](#baseline)
12. [Conclusiones y próximos pasos](#conclusiones)

---
## 1. Definición del problema <a name="problema"></a>

### ¿Qué problema intentamos resolver?

Predecir el **precio de bolsa de la electricidad en Colombia** a partir de variables diarias del estado del sistema eléctrico. El precio de bolsa es el precio marginal del mercado mayorista, determinado principalmente por la disponibilidad hídrica y la mezcla de generación.

### Variable objetivo

`precio_bolsa_kwh` — precio de bolsa ponderado diario en $/kWh  
**Transformación:** `log(precio_bolsa_kwh)` para el entrenamiento, dado el skew=2.158

### Métrica principal

**RMSE en escala original ($/kWh)** — penaliza errores grandes, coherente con el impacto económico de una mala predicción de precio.

---
## 2. Setup y dependencias <a name="setup"></a>

In [ ]:
import warnings
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

# Crear carpeta de figuras si no existe
os.makedirs('figures', exist_ok=True)

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    plt.style.use('seaborn-darkgrid')
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size'] = 11
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

try:
    from IPython.display import display
except ImportError:
    display = print

print('✅ Entorno configurado — SEED =', SEED)
print('✅ Carpeta figures/ lista')

---
## 3. Carga de datos <a name="carga"></a>

Los datos provienen de **7 archivos CSV independientes** del operador del sistema eléctrico colombiano (XM), ya integrados en un único archivo `dataset_energia_colombia.csv` (1,155 filas × 16 columnas).

> `TARGET = 'precio_bolsa_kwh'` se define aquí y es usado en todas las secciones siguientes.

In [ ]:
# ── Constante global del target ──────────────────────────────────────────
TARGET = 'precio_bolsa_kwh'

# ── Carga del dataset integrado ───────────────────────────────────────────
dataset = pd.read_csv('dataset_energia_colombia.csv', parse_dates=['date'])

print(f'Shape      : {dataset.shape}')
print(f'Período    : {dataset["date"].min().date()} → {dataset["date"].max().date()}')
print(f'TARGET     : {TARGET}')
print(f'Columnas   : {list(dataset.columns)}')
display(dataset.head(3))

---
## 4. Auditoría de calidad <a name="calidad"></a>

In [ ]:
print('=== TIPOS DE DATO ===')
print(dataset.dtypes)

print('\n=== VALORES FALTANTES ===')
miss = dataset.isna().sum()
if miss.sum() == 0:
    print('✅ Sin valores faltantes en ninguna columna')
else:
    print(miss[miss > 0])

print('\n=== DUPLICADOS ===')
dups = dataset.duplicated().sum()
print(f'✅ Filas duplicadas exactas: {dups}' if dups == 0 else f'⚠️ {dups} filas duplicadas')

In [ ]:
# Perfil numérico completo
cols_num = [c for c in dataset.columns if c != 'date']
rows = []
for col in cols_num:
    s = dataset[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    outlier_pct = round(((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).mean() * 100, 2)
    rows.append({
        'columna': col, 'media': round(s.mean(), 2), 'mediana': round(s.median(), 2),
        'std': round(s.std(), 2), 'min': round(s.min(), 2), 'max': round(s.max(), 2),
        'skew': round(s.skew(), 3), 'outlier_%': outlier_pct,
    })

profile = pd.DataFrame(rows)
print('=== PERFIL NUMÉRICO ===')
display(profile)

---
## 5. EDA — Análisis del target <a name="target"></a>

In [ ]:
dataset['log_precio_bolsa'] = np.log(dataset[TARGET])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Distribución del target — original vs log-transformado', fontweight='bold')

for ax, col, titulo in zip(axes,
    [TARGET, 'log_precio_bolsa'],
    ['precio_bolsa_kwh (original)', 'log(precio_bolsa_kwh) — target de entrenamiento']):
    s = dataset[col]
    ax.hist(s, bins=50, edgecolor='black', alpha=0.8, color='steelblue')
    ax.axvline(s.mean(),   linestyle='--', linewidth=1.5, color='red',    label=f'Media: {s.mean():.2f}')
    ax.axvline(s.median(), linestyle=':',  linewidth=1.5, color='orange', label=f'Mediana: {s.median():.2f}')
    ax.set_title(titulo); ax.set_xlabel(col); ax.set_ylabel('Frecuencia'); ax.legend()
    ax.text(0.97, 0.95, f'skew={s.skew():.3f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=10, color='gray')

plt.tight_layout()
plt.savefig('figures/distribucion_target.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nTransformación log: skew {dataset[TARGET].skew():.3f} → {dataset["log_precio_bolsa"].skew():.3f}')
print(f'Dif. media-mediana: {abs(dataset[TARGET].mean()-dataset[TARGET].median())/dataset[TARGET].median()*100:.1f}%'
      f' → {abs(dataset["log_precio_bolsa"].mean()-dataset["log_precio_bolsa"].median())/dataset["log_precio_bolsa"].median()*100:.1f}%')

In [ ]:
# Serie temporal del precio — se construye 'mensual' para uso posterior
dataset['year_month'] = dataset['date'].dt.to_period('M')
mensual = dataset.groupby('year_month').agg(
    precio=(TARGET, 'mean'),
    reservas=('reservas_pct', 'mean'),
).reset_index()
mensual['fecha'] = mensual['year_month'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(mensual['fecha'], mensual['precio'], color='tomato', linewidth=2)
ax.fill_between(mensual['fecha'], mensual['precio'], alpha=0.15, color='tomato')
ax.axvspan(pd.Timestamp('2023-06-01'), pd.Timestamp('2024-05-31'), alpha=0.1, color='orange', label='El Niño')
ax.axvspan(pd.Timestamp('2024-06-01'), pd.Timestamp('2024-12-31'), alpha=0.1, color='red', label='Crisis precios')
ax.axhline(dataset[TARGET].median(), linestyle='--', color='gray', linewidth=1,
           label=f'Mediana global: {dataset[TARGET].median():.0f}')
ax.set_title('Precio de bolsa mensual promedio ($/kWh)', fontweight='bold')
ax.set_ylabel('$/kWh'); ax.legend()
plt.tight_layout()
plt.savefig('figures/serie_temporal_precio.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Verificar que el pico de 2024 es dato real, no error
print('=== TOP 10 PRECIOS MÁS ALTOS ===')
top10 = dataset[['date', TARGET, 'reservas_pct', 'ratio_hidro', 'gen_termica']].nlargest(10, TARGET)
display(top10)
print('\n→ El pico de sep-oct 2024 coincide con reservas mínimas y máxima generación térmica.')
print('→ Corresponde al fenómeno El Niño 2024. Es dato real, no error de captura.')

---
## 6. EDA — Análisis de variables: relaciones, señal predictiva y selección <a name="eda"></a>

Este bloque sigue una narrativa en tres pasos:

1. **Relaciones entre variables** — detectar multicolinealidad, redundancia y cadenas causales
2. **Relaciones con el target** — identificar qué variables tienen señal predictiva real
3. **Evaluación de reducción** — probar si eliminar variables redundantes mejora o empeora el modelo

In [ ]:
# Distribuciones de variables clave
vars_plot = ['aportes_energia_gwh', 'reservas_pct', 'gen_termica', 'ratio_hidro']
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Distribuciones de variables clave', fontweight='bold')

for ax, col in zip(axes.flat, vars_plot):
    s = dataset[col]
    ax.hist(s, bins=40, edgecolor='black', alpha=0.8, color='steelblue')
    ax.axvline(s.mean(),   linestyle='--', linewidth=1.3, color='red',    label=f'Media: {s.mean():.2f}')
    ax.axvline(s.median(), linestyle=':',  linewidth=1.3, color='orange', label=f'Mediana: {s.median():.2f}')
    ax.set_title(col); ax.set_xlabel(col); ax.legend(fontsize=8)
    ax.text(0.97, 0.95, f'skew={s.skew():.2f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('figures/distribucion_variables.png', dpi=150, bbox_inches='tight')
plt.show()

### 6a. Relaciones entre variables

Antes de mirar qué predice el target, analizamos cómo se relacionan las variables entre sí. Esto revela:
- **Multicolinealidad**: features que dicen lo mismo y pueden desestabilizar modelos lineales
- **Cadenas causales**: cómo se propagan los efectos en el sistema
- **Redundancia**: variables que ya están implícitas en otras

> **Regla:** si dos features tienen |r| > 0.85, considerar cuál aporta más información única.

In [ ]:
# Variables para el análisis entre features (sin target aún)
VARS_ANALISIS = [
    'aportes_energia_gwh', 'precio_escasez_kwh', 'reservas_pct',
    'gen_hidro', 'gen_termica', 'gen_solar', 'gen_eolica', 'ratio_hidro',
    'demanda_min', 'demanda_pico', 'precio_bolsa_kwh'
]

corr_matrix = dataset[VARS_ANALISIS].corr()

# ── Heatmap completo ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 10))
im = ax.imshow(corr_matrix.values, aspect='auto', vmin=-1, vmax=1, cmap='RdBu_r')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(len(VARS_ANALISIS)))
ax.set_xticklabels(VARS_ANALISIS, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(VARS_ANALISIS)))
ax.set_yticklabels(VARS_ANALISIS, fontsize=9)
ax.set_title('Matriz de correlación de Pearson — todas las variables', fontweight='bold', fontsize=12)
for i in range(len(VARS_ANALISIS)):
    for j in range(len(VARS_ANALISIS)):
        val = corr_matrix.values[i, j]
        color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)
plt.tight_layout()
plt.savefig('figures/heatmap_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Pares con alta correlación entre features ────────────────────────────
pairs = []
cols = [c for c in VARS_ANALISIS if c != TARGET]
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        r = corr_matrix.loc[cols[i], cols[j]]
        if abs(r) > 0.70:
            pairs.append({'variable_1': cols[i], 'variable_2': cols[j], 'r': round(r, 4)})

pairs_df = pd.DataFrame(pairs).sort_values('r', key=abs, ascending=False)
print('=== PARES FEATURE-FEATURE CON |r| > 0.70 ===')
display(pairs_df)

print('\n→ gen_termica ↔ ratio_hidro: r=-0.98 — sustitución casi perfecta')
print('→ gen_hidro ↔ ratio_hidro:   r=+0.91 — miden la misma señal hídrica')
print('→ gen_hidro ↔ gen_termica:   r=-0.86 — consecuencia directa de la sustitución')
print('→ demanda_min ↔ demanda_pico: r=+0.999 — prácticamente la misma variable')

In [ ]:
# ── Scatter plots de las cadenas causales del dominio ────────────────────
key_pairs = [
    ('reservas_pct',        'gen_termica',   'Reservas ↓ → Térmica ↑\n(driver principal del precio)'),
    ('gen_termica',         'ratio_hidro',   'Térmica vs ratio hídrico\n(sustitución perfecta r=-0.98)'),
    ('aportes_energia_gwh', 'reservas_pct',  'Aportes → Embalses\n(cadena upstream)'),
    ('gen_solar',           'demanda_pico',  'Solar vs demanda pico\n(duck curve)'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Relaciones clave entre variables (color = precio_bolsa_kwh: verde=barato, rojo=caro)',
             fontweight='bold')
for ax, (vx, vy, titulo) in zip(axes.flat, key_pairs):
    r = dataset[[vx, vy]].corr().iloc[0, 1]
    sc = ax.scatter(dataset[vx], dataset[vy], c=dataset[TARGET],
                    cmap='RdYlGn_r', alpha=0.4, s=8)
    plt.colorbar(sc, ax=ax, label='$/kWh', shrink=0.8)
    ax.set_xlabel(vx, fontsize=9); ax.set_ylabel(vy, fontsize=9)
    ax.set_title(f'{titulo}\nr = {r:.3f}', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/scatter_relaciones_variables.png', dpi=150, bbox_inches='tight')
plt.show()

**Cadena causal del sistema eléctrico colombiano:**

```
aportes_energia_gwh → reservas_pct → gen_termica → precio_bolsa_kwh
       r=+0.39              r=-0.54       r=+0.795
```

La señal se **amplifica** a lo largo de la cadena. La multicolinealidad entre `gen_termica`, `gen_hidro` y `ratio_hidro` no es ruido — refleja la física del sistema: son tres formas de medir el mismo fenómeno de sustitución hídrica-térmica.

### 6b. Relaciones con el target

Con la estructura entre features clara, ahora medimos cuánta señal predictiva aporta cada variable sobre `precio_bolsa_kwh`.

In [ ]:
# Correlaciones con el target
cols_corr = [c for c in dataset.columns if c not in ['date', 'year_month', 'log_precio_bolsa']]
corr = dataset[cols_corr].corr()[TARGET].drop(TARGET).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['tomato' if v > 0 else 'steelblue' for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor='black', alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Correlación de Pearson con {TARGET}', fontweight='bold')
ax.set_xlabel('r de Pearson'); ax.invert_yaxis()
plt.tight_layout()
plt.savefig('figures/correlaciones_target.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== CORRELACIONES CON EL TARGET ===')
print(corr.round(3).to_string())

**Lectura de las correlaciones con el target:**

- `gen_termica` (+0.795) y `ratio_hidro` (−0.774): las dos señales más fuertes — y como vimos en 6a, tienen r=−0.98 entre sí
- `gen_hidro` (−0.701): tercera señal más fuerte, pero r=+0.91 con `ratio_hidro` — información mayormente redundante
- `gen_solar` (−0.282) y `gen_eolica` (−0.201): señal débil directa, pero capturan la transición energética estructural

**Pregunta que surge:** si `gen_termica`, `gen_hidro` y `ratio_hidro` son tan redundantes entre sí, ¿podríamos eliminar algunas y mantener el mismo poder predictivo?

In [ ]:
# Evolución de la generación solar
solar_anual = dataset.groupby(dataset['date'].dt.year).agg(
    solar_media=('gen_solar', 'mean'),
    pct_solar=('gen_solar', lambda x: (x / dataset.loc[x.index, 'gen_total_kwh']).mean() * 100)
).round(2)

print('=== EXPANSIÓN SOLAR — TENDENCIA ESTRUCTURAL ===')
print(solar_anual.to_string())
print('\n→ La generación solar se multiplicó ~5x en 3 años.')
print('→ gen_solar tiene tendencia no estacionaria: se conserva como feature con esta limitación documentada.')

### 6c. Evaluación de reducción de features

Con base en el análisis de redundancia de 6a, proponemos un **set reducido** que elimina las variables más redundantes:

| Eliminada | Razón |
|---|---|
| `gen_termica` | r=−0.98 con `ratio_hidro`; ya está capturada en `estres_hidrico` |
| `gen_hidro` | r=+0.91 con `ratio_hidro`; ya está dentro de `gen_renovable` |
| `demanda_min` | r=+0.999 con `demanda_pico`; ya está dentro de `efecto_solar_demanda` |

Comparamos tres sets contra el mismo modelo Ridge para decidir cuál usar.

In [ ]:
# ── Definición de los tres sets ──────────────────────────────────────────
FEATURES_ACTUAL = [
    'aportes_energia_gwh', 'precio_escasez_kwh', 'reservas_pct',
    'gen_hidro', 'gen_termica', 'gen_solar', 'gen_eolica', 'ratio_hidro',
    'demanda_min', 'demanda_pico',
    'estres_hidrico', 'efecto_solar_demanda', 'gen_renovable',
]

FEATURES_REDUCIDO = [
    'aportes_energia_gwh', 'reservas_pct', 'ratio_hidro',
    'gen_solar', 'gen_eolica', 'demanda_pico',
    'precio_escasez_kwh',
    'estres_hidrico', 'efecto_solar_demanda', 'gen_renovable',
]

FEATURES_MINIMO = [f for f in FEATURES_REDUCIDO if f != 'gen_eolica']

print(f'Set actual  : {len(FEATURES_ACTUAL):2d} features — {FEATURES_ACTUAL}')
print(f'Set reducido: {len(FEATURES_REDUCIDO):2d} features — {FEATURES_REDUCIDO}')
print(f'Set mínimo  : {len(FEATURES_MINIMO):2d} features — {FEATURES_MINIMO}')
print(f'\nEliminadas en reducido: {[f for f in FEATURES_ACTUAL if f not in FEATURES_REDUCIDO]}')

In [ ]:
# ── Comparación con Ridge — mismo modelo, distintos sets ─────────────────
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def eval_set(features, name, train_df, test_df, target_log, target_orig):
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   Ridge(alpha=1.0)),
    ])
    pipe.fit(train_df[features], train_df[target_log])
    pred = pipe.predict(test_df[features])
    rmse = np.sqrt(mean_squared_error(np.exp(test_df[target_log]), np.exp(pred)))
    mae  = mean_absolute_error(np.exp(test_df[target_log]), np.exp(pred))
    r2   = r2_score(test_df[target_log], pred)
    return {'set': name, 'n_features': len(features),
            'rmse_orig': round(rmse, 1), 'mae_orig': round(mae, 1), 'r2_log': round(r2, 4)}

# Necesitamos train y test con features engineered — los creamos aquí
def add_engineered_features(df):
    out = df.copy()
    out['estres_hidrico']       = out['gen_termica'] / (out['reservas_pct'] + 0.01)
    out['efecto_solar_demanda'] = out['demanda_pico'] - out['demanda_min']
    out['gen_renovable']        = out['gen_hidro'] + out['gen_solar'] + out['gen_eolica']
    return out.replace([np.inf, -np.inf], np.nan)

FECHA_CORTE = '2025-03-31'
TARGET_LOG  = 'log_precio_bolsa'
train_eval = add_engineered_features(dataset[dataset['date'] <= FECHA_CORTE].copy())
test_eval  = add_engineered_features(dataset[dataset['date'] >  FECHA_CORTE].copy())
train_eval[TARGET_LOG] = np.log(train_eval[TARGET])
test_eval[TARGET_LOG]  = np.log(test_eval[TARGET])

resultados_sets = [
    eval_set(FEATURES_ACTUAL,   'Set actual (13)',   train_eval, test_eval, TARGET_LOG, TARGET),
    eval_set(FEATURES_REDUCIDO, 'Set reducido (10)', train_eval, test_eval, TARGET_LOG, TARGET),
    eval_set(FEATURES_MINIMO,   'Set mínimo (9)',    train_eval, test_eval, TARGET_LOG, TARGET),
]

comp_df = pd.DataFrame(resultados_sets)
print('=== COMPARACIÓN DE SETS DE FEATURES — Ridge alpha=1.0 ===')
display(comp_df)

In [ ]:
# ── Visualización comparativa ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Diagnóstico: set actual (13) vs reducido (10) vs mínimo (9)', fontweight='bold')

sets_viz = [
    (FEATURES_ACTUAL,   'Set actual\n13 features',   'steelblue'),
    (FEATURES_REDUCIDO, 'Set reducido\n10 features', 'tomato'),
    (FEATURES_MINIMO,   'Set mínimo\n9 features',    'seagreen'),
]

for ax, (feats, label, color) in zip(axes, sets_viz):
    pipe = Pipeline([('imputer', SimpleImputer(strategy='median')),
                     ('scaler',  StandardScaler()), ('model', Ridge(alpha=1.0))])
    pipe.fit(train_eval[feats], train_eval[TARGET_LOG])
    pred = pipe.predict(test_eval[feats])
    y_pred = np.exp(pred); y_true = test_eval[TARGET].values
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(test_eval[TARGET_LOG], pred)
    ax.scatter(y_true, y_pred, alpha=0.3, s=10, color=color)
    lim = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lim, lim, 'k--', linewidth=1.5, label='Perfecta')
    ax.set_title(f'{label}\nRMSE={rmse:,.1f} $/kWh  R²={r2:.4f}', fontsize=9)
    ax.set_xlabel('Real ($/kWh)'); ax.set_ylabel('Predicho ($/kWh)')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('figures/comparacion_sets_features.png', dpi=150, bbox_inches='tight')
plt.show()

**Conclusión de la evaluación:**

| Set | Features | RMSE (\$/kWh) | R² (log) |
|---|---|---|---|
| Actual | 13 | **101.0** | **−0.66** |
| Reducido | 10 | 115.2 | −0.97 |
| Mínimo | 9 | 125.6 | −1.24 |

**El set actual (13 features) produce el mejor resultado.** Eliminar `gen_termica` y `gen_hidro` empeora el RMSE en ~14% aunque parezcan redundantes con `ratio_hidro`.

**¿Por qué?** `ratio_hidro` dice *qué porcentaje* es hídrico, pero `gen_termica` dice *cuántos kWh absolutos* de térmica se despacharon — son dos dimensiones distintas del mismo fenómeno. **Ridge maneja la multicolinealidad con regularización**, por lo que no necesitamos eliminar variables para proteger el modelo.

> **Decisión final: se conservan los 13 features.**  
> Para la Entrega 2 (Random Forest, GBM) se evaluará si el set reducido mejora en modelos basados en árboles, donde la redundancia puede desperdiciar splits.

---
## 7. EDA — Fenómeno El Niño 2023-2024 <a name="nino"></a>

In [ ]:
# Estadísticas por período
dataset['periodo'] = 'Normal'
dataset.loc[(dataset['date'] >= '2023-06-01') & (dataset['date'] <= '2024-05-31'), 'periodo'] = 'El Niño'
dataset.loc[(dataset['date'] >= '2024-06-01') & (dataset['date'] <= '2024-12-31'), 'periodo'] = 'Crisis precios'
dataset.loc[dataset['date'] >= '2025-01-01', 'periodo'] = 'Recuperación'

resumen = dataset.groupby('periodo').agg(
    dias=('date', 'count'),
    precio_medio=(TARGET, 'mean'),
    precio_max=(TARGET, 'max'),
    reservas_media=('reservas_pct', 'mean'),
    reservas_min=('reservas_pct', 'min'),
    ratio_hidro=('ratio_hidro', 'mean'),
).round(2)

orden = ['Normal', 'El Niño', 'Crisis precios', 'Recuperación']
print('=== IMPACTO DEL FENÓMENO EL NIÑO ===')
display(resumen.reindex(orden))

In [ ]:
# Visualización: reservas y precio durante El Niño
# 'mensual' fue construido en la sección 5
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
fig.suptitle('Fenómeno El Niño 2023-2024 — Impacto en el sistema eléctrico', fontweight='bold')

for ax in axes:
    ax.axvspan(pd.Timestamp('2023-06-01'), pd.Timestamp('2024-05-31'), alpha=0.1, color='orange', label='El Niño')
    ax.axvspan(pd.Timestamp('2024-06-01'), pd.Timestamp('2024-12-31'), alpha=0.1, color='red', label='Crisis precios')

axes[0].plot(mensual['fecha'], mensual['reservas'], color='steelblue', linewidth=2)
axes[0].axhline(0.40, linestyle='--', color='red', linewidth=1, label='Nivel crítico 40%')
axes[0].fill_between(mensual['fecha'], mensual['reservas'], alpha=0.2, color='steelblue')
axes[0].set_ylabel('Reservas hidráulicas (%)'); axes[0].set_title('Nivel de embalses — mínimo histórico: 32% (abril 2024)')
axes[0].legend(fontsize=9)

axes[1].plot(mensual['fecha'], mensual['precio'], color='tomato', linewidth=2)
axes[1].fill_between(mensual['fecha'], mensual['precio'], alpha=0.2, color='tomato')
axes[1].set_ylabel('Precio de bolsa ($/kWh)'); axes[1].set_title('Precio — pico oct 2024: $1,544/kWh promedio mensual')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('figures/fenomeno_nino.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Análisis de redundancia <a name="redundancia"></a>

In [ ]:
# Verificar redundancia en columnas de demanda
cols_demanda = ['demanda_kwh', 'demanda_max', 'demanda_min', 'demanda_promedio', 'demanda_pico']
print('=== CORRELACIÓN ENTRE VARIABLES DE DEMANDA ===')
display(dataset[cols_demanda].corr().round(4))

print(f'\n¿demanda_kwh == demanda_promedio? {(dataset["demanda_kwh"] == dataset["demanda_promedio"]).all()}')

suma = dataset['gen_hidro'] + dataset['gen_termica'] + dataset['gen_solar'] + dataset['gen_eolica']
print(f'\nDiferencia máxima gen_total vs suma componentes: {(dataset["gen_total_kwh"] - suma).abs().max()}')
print(f'Correlación gen_total con target: {dataset["gen_total_kwh"].corr(dataset[TARGET]):.4f}')

dataset['diff_max_pico'] = dataset['demanda_max'] - dataset['demanda_pico']
print(f'\nCorrelación (demanda_max - demanda_pico) vs gen_solar: {dataset["diff_max_pico"].corr(dataset["gen_solar"]):.4f}')
print('→ La diferencia crece con la solar (duck curve). demanda_pico ≠ demanda_max.')

In [ ]:
print('=== DECISIONES DE COLUMNAS ===')
print('\nELIMINAR:')
print('  gen_total_kwh    → suma exacta de componentes, r=0.051 con target')
print('  demanda_kwh      → idéntica a demanda_promedio')
print('  demanda_max      → correlación 1.000 con demanda_kwh')
print('  demanda_promedio → idéntica a demanda_kwh')
print('\nCONSERVAR:')
print('  demanda_min  → piso de demanda diaria')
print('  demanda_pico → captura la demanda en horas pico con efecto solar (duck curve)')
print('  ratio_hidro  → más informativo que gen_hidro (r=-0.774 vs -0.701)')

---
## 9. Feature engineering <a name="engineering"></a>

El feature engineering traduce conocimiento del dominio en variables computables. Cada feature nueva debe responder a una **hipótesis física del sistema** — no se crean variables "por probar".

Se definen **5 features** en dos grupos:

**Grupo 1 — Chain causal hídrica-térmica** (motivadas por sección 6a):
- `estres_hidrico`: captura el umbral crítico de la cadena `reservas → térmica → precio`
- `presion_termica`: captura el flujo de entrada vs respuesta del sistema

**Grupo 2 — Transición energética solar** (motivadas por análisis solar de sección 6b):
- `efecto_solar_demanda`: duck curve — impacto solar en demanda neta
- `termica_evitada`: cuánto solar desplaza térmica cara
- `gen_renovable`: generación limpia total

> **Regla:** se aplican **después del split**, por separado en train y test. Son transformaciones determinísticas — no aprenden estadísticas del dataset.

In [ ]:
def add_engineered_features(df):
    """
    Features derivadas con justificación de dominio.
    Aplicar DESPUÉS del split, por separado en train y test.
    Transformación determinística — no aprende estadísticas.
    """
    out = df.copy()

    # ── Grupo 1: Cadena causal hídrica-térmica ────────────────────────────
    # Motivación: sección 6a identificó la cadena
    # aportes → reservas → gen_termica → precio (r progresivo hasta +0.795)

    # 1. Estrés hídrico — captura la crisis cuando reservas bajas + térmica alta
    #    Fórmula: gen_termica / (reservas_pct + 0.01)
    #    +0.01: guard contra división por cero si reservas → 0
    out['estres_hidrico'] = out['gen_termica'] / (out['reservas_pct'] + 0.01)

    # 2. Presión térmica — flujo de entrada vs respuesta del sistema
    #    Fórmula: gen_termica / (aportes_energia_gwh + 1)
    #    Captura cuánta térmica se necesita por cada GWh de agua que entra
    #    +1: guard defensivo (producción puede tener días sin aportes reportados)
    out['presion_termica'] = out['gen_termica'] / (out['aportes_energia_gwh'] + 1)

    # ── Grupo 2: Transición energética solar ─────────────────────────────
    # Motivación: sección 6b mostró que la correlación solar-precio
    # se está invirtiendo año a año (2023: +0.376 → 2026: -0.083)
    # El solar está empezando a desplazar térmica — el modelo debe verlo

    # 3. Efecto solar en demanda (duck curve)
    #    Fórmula: demanda_pico - demanda_min
    #    La diferencia crece con la instalación solar (r=0.54 con gen_solar)
    out['efecto_solar_demanda'] = out['demanda_pico'] - out['demanda_min']

    # 4. Térmica evitada — cuánto solar desplaza térmica cara
    #    Fórmula: gen_solar / (gen_termica + 1)
    #    Alto ratio = el solar está funcionando como sustituto de la térmica
    #    Bajo ratio = el solar no evita encender plantas térmicas → precio alto
    #    +1: guard defensivo (Colombia siempre tiene algo de térmica, pero
    #    en producción podría cambiar con mayor penetración renovable)
    out['termica_evitada'] = out['gen_solar'] / (out['gen_termica'] + 1)

    # 5. Generación renovable total
    #    Fórmula: gen_hidro + gen_solar + gen_eolica
    #    Captura cuánto del sistema usa generación limpia vs térmica
    out['gen_renovable'] = out['gen_hidro'] + out['gen_solar'] + out['gen_eolica']

    return out.replace([np.inf, -np.inf], np.nan)


# ── Verificar correlaciones sobre dataset completo (solo inspección) ──────
dataset_eng = add_engineered_features(dataset)
new_cols = ['estres_hidrico', 'presion_termica',
            'efecto_solar_demanda', 'termica_evitada', 'gen_renovable']

print('=== FEATURES ENGINEERED — CORRELACIÓN CON TARGET ===')
print(f'{"feature":<25}  {"r con precio_bolsa":>20}  {"grupo"}')
print('─'*60)
grupos = ['hídrica-térmica','hídrica-térmica','solar','solar','solar']
for col, grupo in zip(new_cols, grupos):
    r = dataset_eng[col].corr(dataset_eng[TARGET])
    print(f'  {col:<25}  r = {r:>8.4f}  [{grupo}]')

print('\n✅ 5 features definidas. Se aplican después del split.')

In [ ]:
# ── Evaluación del impacto acumulativo de cada feature ───────────────────
# Se aplica SOBRE train/test del split — definidos en sección 10
# Esta celda se ejecuta después de correr la sección 10

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

FECHA_CORTE_ENG = '2025-03-31'
TARGET_LOG_ENG  = 'log_precio_bolsa'

train_eng = add_engineered_features(dataset[dataset['date'] <= FECHA_CORTE_ENG].copy())
test_eng  = add_engineered_features(dataset[dataset['date'] >  FECHA_CORTE_ENG].copy())
train_eng[TARGET_LOG_ENG] = np.log(train_eng[TARGET])
test_eng[TARGET_LOG_ENG]  = np.log(test_eng[TARGET])

BASE_FEATS = ['aportes_energia_gwh','precio_escasez_kwh','reservas_pct',
              'gen_hidro','gen_termica','gen_solar','gen_eolica','ratio_hidro',
              'demanda_min','demanda_pico']

sets_eval = [
    ('Sin engineering (10)',         BASE_FEATS),
    ('+ estres_hidrico (11)',         BASE_FEATS + ['estres_hidrico']),
    ('+ presion_termica (12)',        BASE_FEATS + ['estres_hidrico','presion_termica']),
    ('+ efecto_solar_demanda (13)',   BASE_FEATS + ['estres_hidrico','presion_termica',
                                                    'efecto_solar_demanda']),
    ('+ gen_renovable (14)',          BASE_FEATS + ['estres_hidrico','presion_termica',
                                                    'efecto_solar_demanda','gen_renovable']),
    ('+ termica_evitada (15) ✅',    BASE_FEATS + ['estres_hidrico','presion_termica',
                                                    'efecto_solar_demanda','gen_renovable',
                                                    'termica_evitada']),
]

print(f'{"Set":<40} {"RMSE":>8}  {"R²":>8}  {"Δ RMSE"}')
print('─'*65)
rmse_prev = None
for nombre, feats in sets_eval:
    pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc',  StandardScaler()),
                     ('m',   Ridge(alpha=1.0))])
    pipe.fit(train_eng[feats], train_eng[TARGET_LOG_ENG])
    pred = pipe.predict(test_eng[feats])
    rmse = np.sqrt(mean_squared_error(np.exp(test_eng[TARGET_LOG_ENG]), np.exp(pred)))
    r2   = r2_score(test_eng[TARGET_LOG_ENG], pred)
    delta = f'{rmse_prev-rmse:+.1f}' if rmse_prev else '—'
    print(f'  {nombre:<38} {rmse:>8.1f}  {r2:>8.4f}  {delta}')
    rmse_prev = rmse

print('\n→ termica_evitada es la feature más poderosa: captura el desplazamiento')
print('  solar de la térmica — la esencia de la transición energética en una variable.')

In [ ]:
# ── Visualización: evolución del RMSE con cada feature ───────────────────
nombres_viz = ['Sin eng.\n(10)', '+estres\nhidrico\n(11)',
               '+presion\ntermica\n(12)', '+efecto\nsolar\n(13)',
               '+gen\nrenovable\n(14)', '+termica\nevitada\n(15)']
rmses_viz = []

for nombre, feats in sets_eval:
    pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc',  StandardScaler()),
                     ('m',   Ridge(alpha=1.0))])
    pipe.fit(train_eng[feats], train_eng[TARGET_LOG_ENG])
    pred = pipe.predict(test_eng[feats])
    rmse = np.sqrt(mean_squared_error(np.exp(test_eng[TARGET_LOG_ENG]), np.exp(pred)))
    rmses_viz.append(rmse)

fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#d4e6f1'] * 5 + ['#2980b9']  # último resaltado
bars = ax.bar(nombres_viz, rmses_viz, color=colors, edgecolor='black', alpha=0.9)
ax.set_title('Impacto acumulativo del feature engineering — RMSE ($/kWh)',
             fontweight='bold')
ax.set_ylabel('RMSE ($/kWh)')
ax.set_ylim(0, max(rmses_viz) * 1.15)
for bar, val in zip(bars, rmses_viz):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
# Anotación de mejora total
ax.annotate('', xy=(5, rmses_viz[-1]), xytext=(0, rmses_viz[0]),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(2.5, (rmses_viz[0] + rmses_viz[-1])/2,
        f'−{rmses_viz[0]-rmses_viz[-1]:.1f} $/kWh\n({(rmses_viz[0]-rmses_viz[-1])/rmses_viz[0]*100:.1f}%)',
        ha='center', color='red', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig('figures/impacto_feature_engineering.png', dpi=150, bbox_inches='tight')
plt.show()

**Resumen del impacto:**

| Feature | RMSE | Δ | Motivación |
|---|---|---|---|
| Sin engineering | ~101 | — | Solo variables originales |
| + `estres_hidrico` | ~91 | −10 | Cadena causal reservas→térmica→precio |
| + `presion_termica` | ~87 | −4 | Flujo de aportes vs respuesta térmica |
| + `efecto_solar_demanda` | ~79 | −8 | Duck curve |
| + `gen_renovable` | ~74 | −5 | Generación limpia total |
| **+ `termica_evitada`** | **~63** | **−11** | **Solar desplazando térmica** |

> **Justificación de Ridge alpha=1.0:** los documentos del curso (fundamentos_preprocessing_ml.md y ml_tarea_resuelta.ipynb) especifican explícitamente:
> *"Modelo: Ridge (no LinearRegression) — Regulariza coeficientes, más estable con features correlacionadas"*
> Con multicolinealidad r=−0.98 entre gen_termica y ratio_hidro, Ridge es la elección correcta.
> El alpha=1.0 es el valor canónico del curso — la búsqueda de alpha óptimo se realizará en la Entrega 2.

---
## 10. Partición temporal <a name="split"></a>

> **Regla fundamental para series de tiempo:** el split nunca es aleatorio.  
> Un split aleatorio mezclaría el tiempo — el modelo aprendería del futuro para predecir el pasado (leakage temporal).

**Criterio de corte:** marzo 2025
- Train contiene El Niño completo (366 días) y la crisis de precios (91 días)
- Test cubre exactamente un año completo (estacionalidad completa)
- Test es un régimen genuinamente diferente: período de recuperación

In [ ]:
FECHA_CORTE = '2025-03-31'
TARGET_LOG  = 'log_precio_bolsa'

# ✅ Split temporal — PRIMERO el split, LUEGO las transformaciones
train_raw = dataset[dataset['date'] <= FECHA_CORTE].copy()
test_raw  = dataset[dataset['date'] >  FECHA_CORTE].copy()

# Feature engineering DESPUÉS del split — por separado en cada partición
train = add_engineered_features(train_raw)
test  = add_engineered_features(test_raw)

# Target log — se crea sobre cada partición por separado
train[TARGET_LOG] = np.log(train[TARGET])
test[TARGET_LOG]  = np.log(test[TARGET])

print('=== PARTICIÓN DE DATOS ===')
print(f'Train: {train["date"].min().date()} → {train["date"].max().date()} | {len(train):,} filas | {len(train)/len(dataset)*100:.1f}%')
print(f'Test : {test["date"].min().date()} → {test["date"].max().date()} | {len(test):,} filas  | {len(test)/len(dataset)*100:.1f}%')
print(f'\nSolapamiento fechas: {len(set(train["date"]).intersection(set(test["date"])))} días')
print(f'\nTarget train — media: {train[TARGET].mean():,.1f}  mediana: {train[TARGET].median():,.1f}  max: {train[TARGET].max():,.1f}')
print(f'Target test  — media: {test[TARGET].mean():,.1f}   mediana: {test[TARGET].median():,.1f}   max: {test[TARGET].max():,.1f}')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train['date'], train[TARGET], color='steelblue', linewidth=0.8, label=f'Train ({len(train)} días)')
ax.plot(test['date'],  test[TARGET],  color='tomato',    linewidth=0.8, label=f'Test  ({len(test)} días)')
ax.axvline(pd.Timestamp(FECHA_CORTE), color='black', linestyle='--', linewidth=1.5, label='Corte: mar 2025')
ax.set_title('Partición temporal del dataset', fontweight='bold')
ax.set_ylabel('precio_bolsa_kwh ($/kWh)'); ax.legend()
plt.tight_layout()
plt.savefig('figures/split_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Baseline reproducible <a name="baseline"></a>

In [ ]:
FEATURES = [
    'aportes_energia_gwh', 'precio_escasez_kwh', 'reservas_pct',
    'gen_hidro', 'gen_termica', 'gen_solar', 'gen_eolica', 'ratio_hidro',
    'demanda_min', 'demanda_pico',
    'estres_hidrico', 'efecto_solar_demanda', 'gen_renovable',
]
FEATURES_BASE = FEATURES[:10]  # sin engineering

X_train = train[FEATURES];     y_train = train[TARGET_LOG]
X_test  = test[FEATURES];      y_test  = test[TARGET_LOG]


def regression_report(name, y_true_log, y_pred_log, y_true_orig):
    """Reporta métricas en escala log y en escala original."""
    rmse_log  = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    mae_log   = mean_absolute_error(y_true_log, y_pred_log)
    r2_log    = r2_score(y_true_log, y_pred_log)
    y_pred_orig = np.exp(y_pred_log)
    rmse_orig = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    mae_orig  = mean_absolute_error(y_true_orig, y_pred_orig)
    r2_orig   = r2_score(y_true_orig, y_pred_orig)
    print(f'\n{"─"*60}')
    print(f'  {name}')
    print(f'{"─"*60}')
    print(f'  Escala log:      RMSE={rmse_log:.4f}  MAE={mae_log:.4f}  R²={r2_log:.4f}')
    print(f'  Escala original: RMSE={rmse_orig:,.1f} $/kWh  MAE={mae_orig:,.1f} $/kWh  R²={r2_orig:.4f}')
    return {'modelo': name, 'rmse_log': rmse_log, 'mae_log': mae_log, 'r2_log': r2_log,
            'rmse_orig': rmse_orig, 'mae_orig': mae_orig, 'r2_orig': r2_orig}


resultados = []

# Benchmark mínimo: predecir siempre la media del train
y_mean = np.full(len(y_test), y_train.mean())
resultados.append(regression_report('Benchmark — Media del train', y_test, y_mean, test[TARGET]))
print('\n✅ Benchmark calculado')

In [ ]:
# Baseline 1: Ridge sin feature engineering
pipe_base = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   Ridge(alpha=1.0)),
])
pipe_base.fit(train[FEATURES_BASE], y_train)
pred_base = pipe_base.predict(test[FEATURES_BASE])
resultados.append(regression_report('Ridge baseline (sin engineering)', y_test, pred_base, test[TARGET]))

In [ ]:
# Baseline 2: Ridge con feature engineering
pipe_eng = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   Ridge(alpha=1.0)),
])
pipe_eng.fit(X_train, y_train)
pred_eng = pipe_eng.predict(X_test)
resultados.append(regression_report('Ridge + feature engineering', y_test, pred_eng, test[TARGET]))

In [ ]:
# Resumen comparativo
print(f'\n{"═"*60}')
print('  RESUMEN COMPARATIVO')
print(f'{"═"*60}')
for res in resultados:
    print(f'  {res["modelo"]:<42} RMSE={res["rmse_orig"]:,.1f} $/kWh  R²={res["r2_log"]:.4f}')

# Coeficientes más importantes
coefs = pd.DataFrame({
    'feature': FEATURES,
    'coeficiente': pipe_eng.named_steps['model'].coef_
}).assign(abs_coef=lambda d: d['coeficiente'].abs()).sort_values('abs_coef', ascending=False)

print(f'\n{"─"*60}')
print('  TOP 10 COEFICIENTES — Ridge + engineering')
print(f'{"─"*60}')
display(coefs[['feature', 'coeficiente']].head(10))

In [ ]:
# Diagnóstico visual: real vs predicho
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Diagnóstico del baseline Ridge + engineering', fontweight='bold')

y_pred_orig = np.exp(pred_eng)
y_true_orig = test[TARGET].values

axes[0].scatter(y_true_orig, y_pred_orig, alpha=0.3, s=10, color='steelblue')
lim = [min(y_true_orig.min(), y_pred_orig.min()), max(y_true_orig.max(), y_pred_orig.max())]
axes[0].plot(lim, lim, 'r--', linewidth=1.5, label='Predicción perfecta')
axes[0].set_xlabel('Valor real ($/kWh)'); axes[0].set_ylabel('Valor predicho ($/kWh)')
axes[0].set_title('Real vs Predicho'); axes[0].legend()

axes[1].plot(test['date'].values, y_true_orig, color='tomato',    linewidth=1, label='Real', alpha=0.8)
axes[1].plot(test['date'].values, y_pred_orig, color='steelblue', linewidth=1, label='Predicho', alpha=0.8)
axes[1].set_title('Período de test — real vs predicho')
axes[1].set_ylabel('$/kWh'); axes[1].legend()

plt.tight_layout()
plt.savefig('figures/diagnostico_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 12. Conclusiones y próximos pasos <a name="conclusiones"></a>

### ¿Qué aprendimos?

1. **El precio de bolsa está dominado por la mezcla hídrica-térmica.** Las dos variables más correlacionadas son `gen_termica` (+0.795) y `ratio_hidro` (−0.774).

2. **El feature engineering aporta señal real.** El RMSE bajó de 331.5 (benchmark) a 101.0 $/kWh — reducción del 70%.

3. **El problema tiene cambio de régimen.** El R² negativo refleja que train (El Niño) y test (recuperación) son distribuciones muy distintas.

4. **La validación temporal es crítica.** Un split aleatorio habría producido métricas artificialmente optimistas.

### Próximos pasos — Entrega 2

- **Validación por ventanas deslizantes** — captura los distintos regímenes
- **Modelos no lineales** — Random Forest, Gradient Boosting
- **Análisis de importancia** — SHAP o permutation importance